<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Daily_Challenge_Pinecone_Serverless_Reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Pinecone Serverless Reranking in Action

In [ ]:
# Part 1 - Install Pinecone libraries
!pip install -q -U pinecone==6.0.1 pinecone-notebooks

In [ ]:
# Authenticate with Pinecone and instantiate the client
import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

In [ ]:
# Define the query and documents
query = "Tell me about Apple's products"

documents = [
    "Apple is a fruit that is often eaten fresh or used in desserts.",
    "Apple develops products such as the iPhone, iPad, Mac, and Apple Watch.",
    "Apples contain fiber, vitamins, and antioxidants.",
    "Apple designs Mac computers and provides services such as iCloud and Apple Music.",
    "The Apple Watch is a wearable device that tracks fitness and health data."
]

# Call the reranker
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[
        {"id": str(i), "text": doc}
        for i, doc in enumerate(documents)
    ],
    top_n=3,
    return_documents=True
)

# Inspect the reranked results
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, match in enumerate(matches):
        print(f"{i + 1}. Score: {match.score}")
        print(f"   Document: {match.document.text}")
        print()

show_reranked_results(query, reranked.data)

In [ ]:
# Part 2 - Install data and model libraries
!pip install -q pandas torch transformers requests

In [ ]:
# Import modules and define environment settings
import os
import time
import pandas as pd
import requests
import tempfile
import torch

from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel

cloud = os.getenv("PINECONE_CLOUD", "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

spec = ServerlessSpec(cloud=cloud, region=region)
index_name = "medical-notes-index"

# Create or recreate the index
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

    while pc.has_index(name=index_name):
        time.sleep(2)

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=spec
)

In [ ]:
# Part 3 - Download and read the JSONL dataset
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = (
        "https://raw.githubusercontent.com/pinecone-io/examples/"
        "refs/heads/main/docs/data/sample_notes_data.jsonl"
    )

    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as file:
        file.write(response.content)

    df = pd.read_json(file_path, orient="records", lines=True)

print("Data shape:", df.shape)
df.head()

In [ ]:
# Part 4 - Upsert the data into Pinecone
index = pc.Index(name=index_name)

index.upsert_from_dataframe(df)

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count:", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

In [ ]:
# Part 5 - Define the embedding function
def get_embedding(input_question):
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    encoded_input = tokenizer(
        input_question,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = model(**encoded_input)

    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

# Run a semantic search query
question = "Which notes describe a patient with knee pain or a knee injury?"
query_vector = get_embedding(question).tolist()

results = index.query(
    vector=query_vector,
    top_k=10,
    include_metadata=True
)

sorted_matches = sorted(
    results["matches"],
    key=lambda match: match["score"],
    reverse=True
)

In [ ]:
# Part 6 - Display the initial search results
def show_results(question, matches):
    print(f"Question: '{question}'")
    print("\nResults:")

    for i, match in enumerate(matches):
        print(f'{str(i + 1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print()

show_results(question, sorted_matches)

In [ ]:
# Prepare the documents for reranking
transformed_documents = [
    {
        "id": match["id"],
        "reranking_field": "; ".join(
            [
                f"{key}: {value}"
                for key, value in match["metadata"].items()
            ]
        )
    }
    for match in results["matches"]
]

# Execute serverless reranking
refined_query = "Which patient needs evaluation or treatment for a knee injury?"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

# Show the reranked results
def show_clinical_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print("\nReranked Results:")

    for i, match in enumerate(matches):
        print(f"{str(i + 1).rjust(4)}. ID: {match.document.id}")
        print(f" Score: {match.score}")
        print(
            f" Reranking Field: "
            f"{match.document.reranking_field}"
        )
        print()

show_clinical_reranked_results(
    refined_query,
    reranked_results.data
)

In [ ]:
# Optional cleanup
# pc.delete_index(name=index_name)